In [1]:
#!pip install implicit

In [2]:
# ============================================================
# CELL 0 — Imports & Config
# ============================================================
import os, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRanker, Pool
import implicit
from tqdm import tqdm

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_sw
from nltk.stem import SnowballStemmer
import re, unicodedata

DATA_DIR    = "/kaggle/input/datasets/artemnazemtsev/nto-ai"
OUTPUT_PATH = "submission.csv"

TOP_K               = 20
CANDIDATES_PER_USER = 150
INCIDENT_DAYS       = 30
HOLDOUT_DAYS        = 30
RANDOM_SEED         = 42

TOP_GENRE_PER_USER  = 10
TOP_AUTHOR_PER_USER = 10
EDS_PER_GENRE       = 40
EDS_PER_AUTHOR      = 30
GLOBAL_POP_CANDS    = 100

ALS_FACTORS = 64
ALS_ITERS   = 30

print("Imports OK")
print(f"Files: {sorted(os.listdir(DATA_DIR))}")


Imports OK
Files: ['authors.csv', 'book_genres.csv', 'editions.csv', 'genres.csv', 'interactions.csv', 'targets.csv', 'users.csv']


In [3]:
# ============================================================
# CELL 1 — Load Data
# ============================================================
interactions = pd.read_csv(
    f"{DATA_DIR}/interactions.csv",
    parse_dates=["event_ts"],
    dtype={"user_id": np.int64, "edition_id": np.int64, "event_type": np.int8},
)
targets     = pd.read_csv(f"{DATA_DIR}/targets.csv",     dtype={"user_id": np.int64})
editions    = pd.read_csv(f"{DATA_DIR}/editions.csv",    dtype={"edition_id": np.int64, "book_id": np.int64, "author_id": np.int64})
book_genres = pd.read_csv(f"{DATA_DIR}/book_genres.csv", dtype={"book_id": np.int64, "genre_id": np.int64})
users       = pd.read_csv(f"{DATA_DIR}/users.csv",       dtype={"user_id": np.int64})
authors     = pd.read_csv(f"{DATA_DIR}/authors.csv",     dtype={"author_id": np.int64})
genres      = pd.read_csv(f"{DATA_DIR}/genres.csv",      dtype={"genre_id": np.int64})

T_MAX            = interactions["event_ts"].max()
T_INCIDENT_START = T_MAX - pd.Timedelta(days=INCIDENT_DAYS)
T_TRAIN_SPLIT    = T_INCIDENT_START - pd.Timedelta(days=HOLDOUT_DAYS)

print(f"interactions : {len(interactions):,}")
print(f"targets      : {len(targets):,}")
print(f"editions     : {len(editions):,}")
print(f"Data range   : {interactions['event_ts'].min().date()} -> {T_MAX.date()}")
print(f"Incident start : {T_INCIDENT_START.date()}")
print(f"Train split    : {T_TRAIN_SPLIT.date()}")


interactions : 443,278
targets      : 3,862
editions     : 460,599
Data range   : 2025-05-01 -> 2025-11-30
Incident start : 2025-10-31
Train split    : 2025-10-01


In [4]:
# ============================================================
# CELL 3 — Feature Engineering Functions
# ============================================================

def build_edition_features(iact: pd.DataFrame, T_incident) -> pd.DataFrame:
    """
    Edition-level features. iact — срез interactions для данного сплита.
    Возвращает DataFrame, индексированный по edition_id.
    """
    pos  = iact[iact["event_type"].isin([1, 2])]

    ep    = pos.groupby("edition_id").size().rename("edition_pop")
    ep_w  = pos[pos["event_type"] == 1].groupby("edition_id").size().rename("wishlist_pop")
    ep_r  = pos[pos["event_type"] == 2].groupby("edition_id").size().rename("read_pop")
    ep_rc = (pos[pos["event_ts"] >= T_incident]
             .groupby("edition_id").size().rename("edition_recent_pop"))
    ep_uu = pos.groupby("edition_id")["user_id"].nunique().rename("edition_unique_users")

    rated = iact[(iact["event_type"] == 2) & iact["rating"].notna()]
    rat   = rated.groupby("edition_id")["rating"].agg(
        rating_mean="mean", rating_std="std", rating_count="count"
    )

    bgc = book_genres.groupby("book_id")["genre_id"].count().rename("book_genre_count")

    feat = editions[[
        "edition_id", "book_id", "author_id",
        "publication_year", "age_restriction", "language_id", "publisher_id",
        "title_clean", "desc_clean",
    ]].copy().set_index("edition_id")

    feat = feat.join(ep).join(ep_w).join(ep_r).join(ep_rc).join(ep_uu).join(rat).join(bgc, on="book_id")

    zero_cols = [
        "edition_pop", "wishlist_pop", "read_pop",
        "edition_recent_pop", "edition_unique_users",
        "rating_mean", "rating_std", "rating_count", "book_genre_count",
    ]
    feat[zero_cols] = feat[zero_cols].fillna(0)

    feat["wishlist_ratio"] = feat["wishlist_pop"] / (feat["edition_pop"] + 1)
    feat["read_ratio"]     = feat["read_pop"]     / (feat["edition_pop"] + 1)
    feat["pop_log"]        = np.log1p(feat["edition_pop"]).astype(np.float32)
    feat["recent_pop_log"] = np.log1p(feat["edition_recent_pop"]).astype(np.float32)

    feat["title_clean"]  = feat["title_clean"].fillna("")
    feat["desc_clean"]   = feat["desc_clean"].fillna("")
    feat["title_len"]    = feat["title_clean"].str.split().str.len().fillna(0).astype(np.float32)
    feat["desc_len"]     = feat["desc_clean"].str.split().str.len().fillna(0).astype(np.float32)
    feat["title_in_desc"]= feat.apply(
        lambda r: float(len(set(r["title_clean"].split()) & set(r["desc_clean"].split()))), axis=1
    ).astype(np.float32)

    f32 = [
        "edition_pop", "wishlist_pop", "read_pop", "edition_recent_pop",
        "edition_unique_users", "rating_mean", "rating_std", "rating_count",
        "book_genre_count", "wishlist_ratio", "read_ratio",
    ]
    feat[f32] = feat[f32].astype(np.float32)
    feat["publication_year"] = feat["publication_year"].fillna(0).astype(np.float32)
    feat["age_restriction"]  = feat["age_restriction"].fillna(0).astype(np.float32)
    feat["language_id"]      = feat["language_id"].fillna(-1).astype(np.int32)
    feat["publisher_id"]     = feat["publisher_id"].fillna(-1).astype(np.int32)
    feat["author_id"]        = feat["author_id"].fillna(-1).astype(np.int64)
    feat["book_id"]          = feat["book_id"].fillna(-1).astype(np.int64)
    return feat


def build_user_features(iact: pd.DataFrame, T_incident) -> pd.DataFrame:
    """User-level features. Возвращает DataFrame, индексированный по user_id."""
    pos = iact[iact["event_type"].isin([1, 2])]

    u_total = pos.groupby("user_id").size().rename("user_activity")
    u_hist  = (pos[pos["event_ts"] <  T_incident].groupby("user_id").size()
               .rename("user_activity_hist"))
    u_inc   = (pos[pos["event_ts"] >= T_incident].groupby("user_id").size()
               .rename("user_activity_inc"))
    u_wish  = (pos[pos["event_type"] == 1].groupby("user_id").size()
               .rename("user_wishlist_count"))
    u_read  = (pos[pos["event_type"] == 2].groupby("user_id").size()
               .rename("user_read_count"))
    u_ue    = pos.groupby("user_id")["edition_id"].nunique().rename("user_uniq_editions")

    rated = iact[(iact["event_type"] == 2) & iact["rating"].notna()]
    u_rat = rated.groupby("user_id")["rating"].agg(
        user_avg_rating="mean", user_rating_count="count"
    )

    feat = users[["user_id", "gender", "age"]].copy().set_index("user_id")
    feat = (feat
            .join(u_total).join(u_hist).join(u_inc)
            .join(u_wish).join(u_read).join(u_ue).join(u_rat))

    zero_cols = [
        "user_activity", "user_activity_hist", "user_activity_inc",
        "user_wishlist_count", "user_read_count", "user_uniq_editions",
        "user_avg_rating", "user_rating_count", "gender", "age",
    ]
    feat[zero_cols] = feat[zero_cols].fillna(0)

    feat["user_wish_ratio"]   = feat["user_wishlist_count"] / (feat["user_activity"] + 1)
    feat["user_read_ratio"]   = feat["user_read_count"]     / (feat["user_activity"] + 1)
    feat["user_activity_log"] = np.log1p(feat["user_activity"])
    feat["user_inc_ratio"]    = feat["user_activity_inc"]   / (feat["user_activity"] + 1)

    return feat.astype(np.float32)


def build_user_genre_profile(iact: pd.DataFrame) -> pd.DataFrame:
    """User × genre affinity. Возвращает плоскую таблицу [user_id, genre_id, user_genre_affinity]."""
    pos = iact[iact["event_type"].isin([1, 2])]
    e2b = editions[["edition_id", "book_id"]].drop_duplicates()
    df  = (pos.merge(e2b, on="edition_id", how="left")
               .merge(book_genres, on="book_id", how="left")
               .dropna(subset=["genre_id"]))
    df["genre_id"] = df["genre_id"].astype(np.int64)

    profile = df.groupby(["user_id", "genre_id"]).size().rename("cnt").reset_index()
    total   = profile.groupby("user_id")["cnt"].transform("sum")
    profile["user_genre_affinity"] = (profile["cnt"] / total).astype(np.float32)
    return profile[["user_id", "genre_id", "user_genre_affinity"]]


def build_user_author_profile(iact: pd.DataFrame) -> pd.DataFrame:
    """User × author affinity. Возвращает плоскую таблицу [user_id, author_id, user_author_affinity]."""
    pos = iact[iact["event_type"].isin([1, 2])]
    e2a = editions[["edition_id", "author_id"]].drop_duplicates()
    df  = pos.merge(e2a, on="edition_id", how="left").dropna(subset=["author_id"])
    df["author_id"] = df["author_id"].astype(np.int64)

    profile = df.groupby(["user_id", "author_id"]).size().rename("cnt").reset_index()
    total   = profile.groupby("user_id")["cnt"].transform("sum")
    profile["user_author_affinity"] = (profile["cnt"] / total).astype(np.float32)
    return profile[["user_id", "author_id", "user_author_affinity"]]


print("Feature functions OK")


Feature functions OK


In [ ]:
# ============================================================
# CELL 2 — Text Preprocessing (OPTIMIZED)
# ============================================================

# 1. Устанавливаем и инициализируем pandarallel для использования всех ядер CPU
!pip install pandarallel -q
from pandarallel import pandarallel
# Инициализация (progress_bar=True показывает прогресс-бар)
pandarallel.initialize(progress_bar=True, nb_workers=os.cpu_count())

# 2. Настраиваем NLTK (глобально, чтобы это было видно всем процессам)
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_sw
from nltk.stem import SnowballStemmer
import re, unicodedata

RU_STOP   = set(nltk_sw.words("russian"))
EN_STOP   = set(nltk_sw.words("english"))
STOPWORDS = RU_STOP | EN_STOP
STEMMER   = SnowballStemmer("russian")

# Регулярные выражения
_RE_HTML  = re.compile(r"<[^>]+>")
_RE_CHARS = re.compile(r"[^а-яёa-z\s]")

# 3. Функция для тяжелой части: стемминг и фильтрация (будет параллелиться)
def stem_and_filter(text: str, max_tokens: int = 120) -> str:
    """
    Получает уже очищенный текст, делает стемминг и отсечку.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    tokens = [
        STEMMER.stem(t)
        for t in text.split()
        if len(t) >= 3 and t not in STOPWORDS
    ]
    return " ".join(tokens[:max_tokens])

# 4. Функция для легкой части: очистка (выполняется векторизовано, очень быстро)
def vector_clean(series: pd.Series) -> pd.Series:
    """
    Быстрая очистка через встроенные методы Pandas (работают на C).
    """
    s = series.fillna("").astype(str)
    s = s.str.normalize("NFKC")               # Нормализация (ё -> е и т.д.)
    s = s.str.replace(_RE_HTML, " ", regex=True) # Удаление HTML
    s = s.str.lower()                         # Нижний регистр
    s = s.str.replace(_RE_CHARS, " ", regex=True) # Удаление мусорных символов
    return s

# 5. Применяем
# Сначала быстрая векторизованная очистка для всего столбца
print("Cleaning text (Vectorized)...")
t_prep = vector_clean(editions["title"])
d_prep = vector_clean(editions["description"])

# Затем параллельный стемминг (самый долгий этап разбит на все ядра)
print("Stemming titles (Parallel)...")
editions["title_clean"] = t_prep.parallel_apply(stem_and_filter)

print("Stemming descriptions (Parallel)...")
# Для описаний передаем max_tokens=120 через lambda
editions["desc_clean"] = d_prep.parallel_apply(lambda x: stem_and_filter(x, max_tokens=120))

# Средний рейтинг изданий (без изменений)
ed_avg_rating_map = (
    interactions[interactions["event_type"] == 2]
    .groupby("edition_id")["rating"].mean()
    .fillna(0).to_dict()
)

print(f"Пример title_clean : {editions['title_clean'].iloc[5]}")
print(f"Пример desc_clean  : {editions['desc_clean'].iloc[5][:150]}")

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Cleaning text (Vectorized)...
Stemming titles (Parallel)...


Stemming descriptions (Parallel)...


In [ ]:
# ============================================================
# CELL 4 — ALS Training (WITH TIME DECAY)
# ============================================================
def train_als_with_decay(iact: pd.DataFrame, half_life_days=30):
    pos = iact[iact["event_type"].isin([1, 2])].copy()
    
    pos["base_w"] = pos["event_type"].map({1: 1.0, 2: 2.0})
    
    pos['event_ts'] = pd.to_datetime(pos['event_ts'])
    max_date = pos['event_ts'].max()
    pos['days_old'] = (max_date - pos['event_ts']).dt.days.clip(lower=0)
    
    pos['decay_weight'] = np.exp(-np.log(2) * pos['days_old'] / half_life_days)
    pos["w"] = (pos["base_w"] * pos['decay_weight']).astype(np.float32)

    ue_enc = LabelEncoder()
    ei_enc = LabelEncoder()
    u_idx  = ue_enc.fit_transform(pos["user_id"])
    e_idx  = ei_enc.fit_transform(pos["edition_id"])

    mat = csr_matrix((pos["w"].values, (u_idx, e_idx)), shape=(len(ue_enc.classes_), len(ei_enc.classes_)))

    als = implicit.als.AlternatingLeastSquares(factors=64, iterations=15, use_gpu=True, random_state=42)
    als.fit(mat) 

    U = als.user_factors.to_numpy().astype(np.float32)
    V = als.item_factors.to_numpy().astype(np.float32)

    U_normed = U / np.linalg.norm(U, axis=1, keepdims=True).clip(min=1e-9)
    V_normed = V / np.linalg.norm(V, axis=1, keepdims=True).clip(min=1e-9)

    u2als = {uid: i for i, uid in enumerate(ue_enc.classes_)}
    e2als = {eid: i for i, eid in enumerate(ei_enc.classes_)}

    # ВАЖНО: Возвращаем объект als первым элементом
    return als, U, V, U_normed, V_normed, u2als, e2als, ue_enc.classes_, ei_enc.classes_

als_model, U, V, U_normed, V_normed, u2als, e2als, als_user_ids, als_edition_ids = train_als_with_decay(interactions)

In [ ]:
# ============================================================
# CELL 5 — ALS Feature Builder (векторизованный)
# ============================================================
ALS_FACTOR_COLS = [f"als_u_{i}" for i in range(ALS_FACTORS)] +  [f"als_v_{i}" for i in range(ALS_FACTORS)]
ALS_SCALAR_COLS = [
    "als_dot", "als_cos",
    "als_nb_mean_sim", "als_nb_max_sim",
    "als_nb_mean_rating", "als_nb_weighted_rating", "als_nb_read_ratio",
]
ALS_FEAT_COLS = ALS_FACTOR_COLS + ALS_SCALAR_COLS


def build_als_features(df_cands: pd.DataFrame, iact: pd.DataFrame) -> pd.DataFrame:
    """
    Для каждой пары (user, edition) в df_cands вычисляет:
      • 64 user-факторов + 64 item-факторов из ALS
      • als_dot  — скалярное произведение u·v
      • als_cos  — косинусное сходство u·v (нормированные)
      • als_nb_*  — агрегаты по топ-10 ближайшим книгам из истории пользователя
                    в ALS item-пространстве

    iact используется для построения истории пользователя (не течёт метка).
    """
    n = len(df_cands)
    df_r = df_cands.reset_index(drop=True)

    # ── Индексы для всех строк ───────────────────────────────────────────────
    u_ser = df_r["user_id"].map(u2als)       # NaN если нет в ALS
    e_ser = df_r["edition_id"].map(e2als)

    valid_u = u_ser.notna().values
    valid_e = e_ser.notna().values
    valid_ue = valid_u & valid_e

    u_idxs = np.where(valid_u, u_ser.fillna(0).astype(int).values, 0)
    e_idxs = np.where(valid_e, e_ser.fillna(0).astype(int).values, 0)

    # ── Факторы (батчевый lookup) ────────────────────────────────────────────
    u_fac = U[u_idxs].copy();  u_fac[~valid_u] = 0.0
    v_fac = V[e_idxs].copy();  v_fac[~valid_e] = 0.0

    # ── Dot / cosine ─────────────────────────────────────────────────────────
    dots = (u_fac * v_fac).sum(axis=1).astype(np.float32)

    un = U_normed[u_idxs].copy(); un[~valid_u] = 0.0
    vn = V_normed[e_idxs].copy(); vn[~valid_e] = 0.0
    coss = (un * vn).sum(axis=1).astype(np.float32)
    coss[~valid_ue] = 0.0

    # ── История пользователя: user_id → массив индексов ALS-items ───────────
    pos_iact = iact[iact["event_type"].isin([1, 2])]
    read_pairs = set(zip(
        pos_iact[pos_iact["event_type"] == 2]["user_id"],
        pos_iact[pos_iact["event_type"] == 2]["edition_id"],
    ))
    user_hist_idx = {}
    for uid, grp in pos_iact.groupby("user_id"):
        idxs = np.array(
            [e2als[eid] for eid in grp["edition_id"] if eid in e2als], dtype=np.int32
        )
        if len(idxs) > 0:
            user_hist_idx[uid] = idxs

    # ── Агрегаты по топ-10 соседям (цикл по пользователям, не по строкам) ───
    nb_mean_sim     = np.zeros(n, np.float32)
    nb_max_sim      = np.zeros(n, np.float32)
    nb_mean_rating  = np.zeros(n, np.float32)
    nb_weighted_rat = np.zeros(n, np.float32)
    nb_read_ratio   = np.zeros(n, np.float32)

    for uid, grp_df in df_r.groupby("user_id"):
        hist = user_hist_idx.get(uid)
        if hist is None or len(hist) == 0:
            continue

        row_idxs  = grp_df.index.values
        cand_eidxs = e_idxs[row_idxs]
        cand_valid = valid_e[row_idxs]

        if not cand_valid.any():
            continue

        # (n_cands, n_hist) косинусное сходство
        sim_mat = V_normed[cand_eidxs] @ V_normed[hist].T  # (n_cands, n_hist)
        sim_mat[~cand_valid] = 0.0

        k10 = min(10, sim_mat.shape[1])
        top_k = np.argpartition(sim_mat, -k10, axis=1)[:, -k10:]  # (n_cands, k10)

        hist_eids = als_edition_ids[hist]  # оригинальные edition_id

        for li, row_i in enumerate(row_idxs):
            if not cand_valid[li]:
                continue
            tidx    = top_k[li]
            tsims   = sim_mat[li, tidx]
            teids   = hist_eids[tidx]
            ratings = np.array([ed_avg_rating_map.get(eid, 0.0) for eid in teids], np.float32)
            reads   = np.array([(uid, eid) in read_pairs for eid in teids], np.float32)
            w       = np.clip(tsims, 0, None) + 1e-9

            nb_mean_sim[row_i]     = tsims.mean()
            nb_max_sim[row_i]      = tsims.max()
            nb_mean_rating[row_i]  = ratings.mean()
            nb_weighted_rat[row_i] = np.average(ratings, weights=w)
            nb_read_ratio[row_i]   = reads.mean()

    # ── Собираем DataFrame ───────────────────────────────────────────────────
    out = {f"als_u_{i}": u_fac[:, i] for i in range(ALS_FACTORS)}
    out.update({f"als_v_{i}": v_fac[:, i] for i in range(ALS_FACTORS)})
    out["als_dot"]               = dots
    out["als_cos"]               = coss
    out["als_nb_mean_sim"]       = nb_mean_sim
    out["als_nb_max_sim"]        = nb_max_sim
    out["als_nb_mean_rating"]    = nb_mean_rating
    out["als_nb_weighted_rating"]= nb_weighted_rat
    out["als_nb_read_ratio"]     = nb_read_ratio

    return pd.DataFrame(out, index=df_cands.index)


print("ALS feature builder OK")
print(f"ALS_FEAT_COLS ({len(ALS_FEAT_COLS)}): {ALS_SCALAR_COLS}")


In [ ]:
# ============================================================
# CELL 6 — Candidate Generation (Full Integrated Version)
# ============================================================

def _genre_index(edition_feat: pd.DataFrame) -> dict:
    # Группируем издания по жанрам, сортируя по популярности внутри жанра
    bg = book_genres.merge(
        edition_feat[["book_id", "pop_log"]].reset_index(), on="book_id", how="inner"
    ).sort_values(["genre_id", "pop_log"], ascending=[True, False])
    return (bg.groupby("genre_id")["edition_id"]
              .apply(lambda x: x.head(EDS_PER_GENRE).tolist())
              .to_dict())

def _author_index(edition_feat: pd.DataFrame) -> dict:
    # Группируем издания по авторам, сортируя по популярности
    tmp = edition_feat[["author_id", "pop_log"]].reset_index()
    tmp = tmp.sort_values(["author_id", "pop_log"], ascending=[True, False])
    return (tmp.groupby("author_id")["edition_id"]
               .apply(lambda x: x.head(EDS_PER_AUTHOR).tolist())
               .to_dict())
'''
def generate_candidates(
    user_ids,
    iact: pd.DataFrame,
    edition_feat: pd.DataFrame,
    user_genre_profile: pd.DataFrame,
    user_author_profile: pd.DataFrame,
    als_model, # Объект implicit.als.AlternatingLeastSquares
    u2als, 
    e2als, 
    als_edition_ids,
    k: int = CANDIDATES_PER_USER,
) -> pd.DataFrame:
    
    print("Initializing indexes...")
    genre_idx  = _genre_index(edition_feat)
    author_idx = _author_index(edition_feat)
    
    # Сет уже виденных (позитивных) пар, чтобы не предлагать их снова
    seen = set(zip(
        iact.loc[iact["event_type"].isin([1, 2]), "user_id"],
        iact.loc[iact["event_type"].isin([1, 2]), "edition_id"],
    ))

    # Топ-N глобально популярных книг
    global_top = (edition_feat["pop_log"]
                  .sort_values(ascending=False)
                  .head(GLOBAL_POP_CANDS).index.tolist())

    # Профили интересов юзеров
    ugp = (user_genre_profile
           .sort_values(["user_id", "user_genre_affinity"], ascending=[True, False])
           .groupby("user_id")
           .apply(lambda g: list(g["genre_id"]))
           .to_dict())

    uap = (user_author_profile
           .sort_values(["user_id", "user_author_affinity"], ascending=[True, False])
           .groupby("user_id")
           .apply(lambda g: list(g["author_id"]))
           .to_dict())

    # История для I2I (Item-to-Item): берем последние 5 айтемов
    pos_iact = iact[iact["event_type"].isin([1, 2])].sort_values('event_ts')
    user_history = pos_iact.groupby('user_id')['edition_id'].apply(lambda x: list(x)[-5:]).to_dict()

    # --- ПРЕДРАСЧЕТ ALS U2I (User-to-Item) ---
    # Делаем батчами через матричное умножение для скорости
    u_idxs = np.array([u2als.get(u, -1) for u in user_ids])
    valid_mask = u_idxs != -1
    valid_u_idxs = u_idxs[valid_mask]
    valid_uids   = np.array(user_ids)[valid_mask]

    als_cands_map = {} 
    
    if len(valid_u_idxs) > 0:
        # Извлекаем факторы напрямую из модели (они уже в numpy после нашего фикса в Cell 4)
        U = als_model.user_factors.to_numpy()
        V = als_model.item_factors.to_numpy()
        u_vecs = U[valid_u_idxs]
        
        batch_size = 500 
        als_top_n = 100 # Сколько брать из ALS U2I
        
        print(f"Calculating ALS U2I for {len(valid_uids)} users...")
        for i in range(0, len(u_vecs), batch_size):
            batch_u = u_vecs[i:i+batch_size]
            batch_uids = valid_uids[i:i+batch_size]
            
            # Считаем скоры для всех айтемов
            scores = batch_u @ V.T
            # Находим индексы лучших через partition (быстрее полной сортировки)
            ind = np.argpartition(scores, -als_top_n, axis=1)[:, -als_top_n:]
            
            for row_idx, uid in enumerate(batch_uids):
                row_scores = scores[row_idx, ind[row_idx]]
                sorted_local_idx = np.argsort(-row_scores)
                top_item_indices = ind[row_idx, sorted_local_idx]
                als_cands_map[uid] = [als_edition_ids[idx] for idx in top_item_indices]

    # --- ОСНОВНОЙ ЦИКЛ СБОРА КАНДИДАТОВ ---
    rows = []
    for uid in tqdm(user_ids, desc="Processing users"):
        cands = {} # eid -> [total_score, src_genre, src_author, src_global, src_u2i, src_i2i]

        def add(eid, score_add, src_idx):
            if (uid, eid) in seen:
                return
            if eid not in cands:
                cands[eid] = [0.0, 0, 0, 0, 0, 0]
            
            cands[eid][0] += score_add
            cands[eid][src_idx] = 1

        # 1. ALS User-to-Item (U2I)
        for rank, eid in enumerate(als_cands_map.get(uid, [])):
            # Используем Reciprocal Rank для нормировки скоров
            add(eid, 4.0 / (rank + 1), 4)

        # 2. ALS Item-to-Item (I2I)
        u_hist = user_history.get(uid, [])
        for hist_eid in u_hist:
            if hist_eid not in e2als: continue
            
            # Получаем похожие айтемы через встроенный метод implicit
            # Он эффективен на GPU
            ids, sim_scores = als_model.similar_items(e2als[hist_eid], N=15)
            
            for rank, (idx, sim) in enumerate(zip(ids, sim_scores)):
                eid = als_edition_ids[idx]
                if eid == hist_eid: continue
                # Вес зависит от косинусной похожести и позиции
                add(eid, (3.0 * sim) / (rank + 1), 5)

        # 3. Жанры (на основе профиля пользователя)
        genre_rank = 0
        for gid in ugp.get(uid, [])[:TOP_GENRE_PER_USER]:
            for eid in genre_idx.get(gid, []):
                add(eid, 1.5 / (genre_rank + 1), 1)
                genre_rank += 1

        # 4. Авторы (на основе профиля пользователя)
        author_rank = 0
        for aid in uap.get(uid, [])[:TOP_AUTHOR_PER_USER]:
            for eid in author_idx.get(aid, []):
                add(eid, 1.5 / (author_rank + 1), 2)
                author_rank += 1

        # 5. Глобальный топ (для «холодных» пользователей)
        for rank, eid in enumerate(global_top):
            add(eid, 0.5 / (rank + 1), 3)

        # Сохраняем топ-K кандидатов для этого пользователя
        sorted_cands = sorted(cands.items(), key=lambda x: -x[1][0])[:k]
        for eid, info in sorted_cands:
            rows.append({
                "user_id": uid, 
                "edition_id": eid,
                "cand_score": info[0],
                "src_genre": info[1], 
                "src_author": info[2], 
                "src_global": info[3], 
                "src_als_u2i": info[4],
                "src_als_i2i": info[5],
            })

    df = pd.DataFrame(rows)
    # Оптимизация типов данных для экономии памяти
    df["cand_score"] = df["cand_score"].astype(np.float32)
    source_cols = ["src_genre", "src_author", "src_global", "src_als_u2i", "src_als_i2i"]
    df[source_cols] = df[source_cols].astype(np.int8)
    
    print(f"Total candidates: {len(df):,}")
    return df

print("Full Candidate Generation logic is ready.")
'''

In [ ]:
import gc
import numpy as np
from tqdm.auto import tqdm

def generate_candidates(targets, model, user_item_matrix, user_mapping, item_mapping, 
                       pop_items, u2i_N=150, batch_size=2000):
    """
    Быстрая и векторизованная генерация U2I кандидатов с фоллбеком на популярное.
    """
    print(f"Генерация кандидатов для {len(targets)} пользователей...")
    
    # Словари для быстрого маппинга
    # Превращаем item_mapping в pandas Series для быстрого векторизованного маппинга
    item_map_s = pd.Series(item_mapping)
    
    all_u2i_dfs = []
    
    for start_idx in tqdm(range(0, len(targets), batch_size), desc="U2I Candidates"):
        batch_users = targets[start_idx : start_idx + batch_size]
        
        # Получаем внутренние ID юзеров. Если юзера нет в маппинге, ставим -1 (обрабатываем ниже)
        user_indices = [user_mapping.get(u, -1) for u in batch_users]
        valid_user_mask = np.array(user_indices) != -1
        
        valid_batch_users = np.array(batch_users)[valid_user_mask]
        valid_user_indices = np.array(user_indices)[valid_user_mask]
        
        if len(valid_user_indices) == 0:
            continue
            
        # 1. ALS Recommendations (сразу матрицей batch_size x N)
        ids, scores = model.recommend(
            valid_user_indices, 
            user_item_matrix[valid_user_indices], 
            N=u2i_N, 
            filter_already_liked_items=True
        )
        
        # 2. Быстрое "сплющивание" (flatten) матриц в колонки
        batch_users_rep = np.repeat(valid_batch_users, ids.shape[1])
        ids_flat = ids.flatten()
        scores_flat = scores.flatten()
        
        # Отфильтровываем пустые предсказания (implicit возвращает -1)
        valid_items_mask = ids_flat != -1
        
        # 3. Собираем батч в DataFrame
        batch_df = pd.DataFrame({
            'user_id': batch_users_rep[valid_items_mask],
            'internal_id': ids_flat[valid_items_mask],
            'score_u2i': scores_flat[valid_items_mask]
        })
        
        # Переводим внутренние ID в реальные edition_id
        batch_df['edition_id'] = batch_df['internal_id'].map(item_map_s)
        batch_df = batch_df.drop(columns=['internal_id'])
        
        all_u2i_dfs.append(batch_df)
    
    # Склеиваем всё вместе
    final_cands = pd.concat(all_u2i_dfs, ignore_index=True) if all_u2i_dfs else pd.DataFrame(columns=['user_id', 'edition_id', 'score_u2i'])
    del all_u2i_dfs
    gc.collect()
    
    # Обработка холодных юзеров (кого не было в матрице)
    # Раздаем им глобальные популярные книги
    missing_users = set(targets) - set(final_cands['user_id'].unique())
    if missing_users:
        print(f"Добавляем популярные товары для {len(missing_users)} холодных пользователей...")
        pop_cands = pd.DataFrame({
            'user_id': np.repeat(list(missing_users), len(pop_items)),
            'edition_id': np.tile(pop_items, len(missing_users)),
            'score_u2i': 0.0 # для популярного скор ALS равен 0
        })
        final_cands = pd.concat([final_cands, pop_cands], ignore_index=True)
    
    # Удаляем дубликаты на случай, если что-то задвоилось
    final_cands = final_cands.drop_duplicates(subset=['user_id', 'edition_id'])
    
    print(f"Сгенерировано {len(final_cands)} кандидатов.")
    return final_cands

In [ ]:
# ============================================================
# CELL 7 — Feature Lists + build_ranking_dataset
# ============================================================
import gc

NUMERIC_FEATURES = [
    # candidate
    "cand_score", "src_genre", "src_author", "src_global", "src_als_u2i", "src_als_i2i",
    # edition counters
    "edition_pop", "wishlist_pop", "read_pop",
    "edition_recent_pop", "edition_unique_users",
    "rating_mean", "rating_std", "rating_count",
    "wishlist_ratio", "read_ratio",
    "pop_log", "recent_pop_log",
    "publication_year", "age_restriction",
    "language_id", "publisher_id",
    "book_genre_count",
    "title_len", "desc_len", "title_in_desc",
    # user counters
    "gender", "age",
    "user_activity", "user_activity_hist", "user_activity_inc",
    "user_wishlist_count", "user_read_count", "user_uniq_editions",
    "user_avg_rating", "user_rating_count",
    "user_wish_ratio", "user_read_ratio",
    "user_activity_log", "user_inc_ratio",
    # cross user×item
    "user_genre_affinity", "user_author_affinity",
] + ALS_FEAT_COLS

TEXT_FEATURES = ["title_clean", "desc_clean"]
ALL_FEATURES = NUMERIC_FEATURES + TEXT_FEATURES

def build_ranking_dataset(cands_df, u_feat, ed_feat, user_genre_profile=None, user_author_profile=None, editions_info=None, iact_hist=None, **kwargs):
    print(f"Начальный размер кандидатов: {cands_df.shape}")
    
    # 1. Используем join вместо merge. 
    # Так как u_feat и ed_feat индексированы по user_id и edition_id, это работает в разы быстрее и без лишних аллокаций памяти.
    df = cands_df.join(u_feat, on='user_id')
    df = df.join(ed_feat, on='edition_id')

    # 2. Подтягиваем author_id и book_id
    if editions_info is not None:
        cols_to_merge = ['edition_id', 'author_id', 'book_id']
        cols_to_merge = [c for c in cols_to_merge if c not in df.columns or c == 'edition_id']
        if len(cols_to_merge) > 1:
            ed_sub = editions_info[cols_to_merge].set_index('edition_id')
            df = df.join(ed_sub, on='edition_id')

    # 3. У профилей составные ключи, оставляем merge, но они лёгкие
    if user_author_profile is not None:
        join_keys = [c for c in ['user_id', 'author_id', 'edition_id'] if c in df.columns and c in user_author_profile.columns]
        if join_keys:
            df = df.merge(user_author_profile, on=join_keys, how='left')
            
    if user_genre_profile is not None:
        join_keys = [c for c in ['user_id', 'book_id', 'edition_id'] if c in df.columns and c in user_genre_profile.columns]
        if join_keys:
            df = df.merge(user_genre_profile, on=join_keys, how='left')

    # 4. Обработка истории взаимодействий (ALS-фичи)
    if iact_hist is not None:
        print("  -> Вычисляем ALS-фичи...")
        als_feats = build_als_features(df, iact_hist)
        # ФИКС: Склеиваем по столбцам, а не перезаписываем df!
        df = pd.concat([df, als_feats], axis=1)
        del als_feats
        gc.collect()

    # 5. Обработка пропусков и даункаст типов (строго In-Place)
    print("  -> Оптимизация памяти In-Place...")
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col].fillna('', inplace=True)
        else:
            df[col].fillna(0, inplace=True)
            # Принудительно сжимаем float64 до float32 (экономия 50% RAM на числах)
            if df[col].dtype == 'float64':
                df[col] = df[col].astype(np.float32)
                
    print(f"Размер итогового датасета: {df.shape}")
    return df

In [ ]:
# ============================================================
# CELL 9 — Безопасная сборка фичей (без взрыва памяти)
# ============================================================

def build_ranking_dataset(cands_df, u_feat, ed_feat, user_genre_profile=None, 
                          user_author_profile=None, editions_info=None, book_genres_df=None, **kwargs):
    
    print(f"Начальный размер кандидатов: {cands_df.shape}")
    
    # 1. Базовые фичи
    df = cands_df.merge(u_feat, on='user_id', how='left')
    df = df.merge(ed_feat, on='edition_id', how='left')

    # 2. Подтягиваем author_id и book_id
    if editions_info is not None:
        cols = [c for c in ['edition_id', 'author_id', 'book_id'] if c not in df.columns or c == 'edition_id']
        if len(cols) > 1:
            df = df.merge(editions_info[cols], on='edition_id', how='left')

    # 3. Авторский профиль (безопасный джоин, так как у книги обычно 1 автор)
    if user_author_profile is not None and 'author_id' in df.columns:
        df = df.merge(user_author_profile, on=['user_id', 'author_id'], how='left')
            
    # 4. Жанровый профиль (УМНАЯ АГРЕГАЦИЯ БЕЗ ВЗРЫВА)
    if user_genre_profile is not None and book_genres_df is not None and 'book_id' in df.columns:
        print("Подклеиваем жанры (агрегация)...")
        # Берем только те пары юзер-книга, которые есть в кандидатах
        user_books = df[['user_id', 'book_id']].drop_duplicates()
        
        # Узнаем жанры этих книг
        ub_genres = user_books.merge(book_genres_df, on='book_id', how='inner')
        
        # Джоиним веса жанров для конкретного юзера
        ub_weights = ub_genres.merge(user_genre_profile, on=['user_id', 'genre_id'], how='inner')
        
        # СУММИРУЕМ веса жанров по каждой книге, чтобы получить 1 строку на пару (user_id, book_id)
        ub_agg = ub_weights.groupby(['user_id', 'book_id'])['genre_weight'].sum().reset_index()
        
        # Приклеиваем обратно к основному датасету (теперь это 1 к 1)
        df = df.merge(ub_agg, on=['user_id', 'book_id'], how='left')

    # 5. Чистка пропусков
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('')
        else:
            df[col] = df[col].fillna(0)
            
    print(f"Размер итогового датасета: {df.shape}")
    return df


In [ ]:
# ============================================================
# CELL 8 — CatBoost: train & predict
# ============================================================

def _has_gpu() -> bool:
    try:
        import subprocess
        return subprocess.run(["nvidia-smi"], capture_output=True, timeout=5).returncode == 0
    except Exception:
        return False


def train_catboost(rdf: pd.DataFrame, labels: np.ndarray) -> CatBoostRanker:
    """
    Обучает CatBoostRanker (YetiRank).
    Группы без позитивов или без негативов отфильтровываются.
    """
    df = rdf.copy()
    df["label"] = labels.astype(np.float32)

    gpos  = df.groupby("user_id")["label"].sum()
    gcnt  = df.groupby("user_id")["label"].count()
    valid = gpos[(gpos >= 1) & (gpos < gcnt)].index
    df    = df[df["user_id"].isin(valid)].sort_values("user_id").reset_index(drop=True)

    print(f"  Training groups : {df['user_id'].nunique():,}")
    print(f"  Positives       : {int(df['label'].sum()):,} / {len(df):,}")

    feat_cols  = [c for c in NUMERIC_FEATURES if c in df.columns]
    text_cols  = [c for c in TEXT_FEATURES    if c in df.columns]

    pool = Pool(
        data=df[feat_cols + text_cols],
        label=df["label"].values,
        group_id=df["user_id"].values,
        text_features=text_cols,
        feature_names=feat_cols + text_cols,
    )

    use_gpu = _has_gpu()
    model = CatBoostRanker(
        loss_function="YetiRank",
        iterations=700,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5,
        random_seed=RANDOM_SEED,
        verbose=100,
        task_type="GPU" if use_gpu else "CPU",
        text_processing={
            "tokenizers": [
                {"tokenizer_id": "Space", "delimiter": " ", "lowercasing": "true"}
            ],
            "dictionaries": [
                {"dictionary_id": "Word",   "max_dictionary_size": "50000", "gram_order": "1"},
                {"dictionary_id": "BiGram", "max_dictionary_size": "50000", "gram_order": "2"},
            ],
            "feature_processing": {
                "default": [
                    {"dictionaries_names": ["Word"],
                     "feature_calcers": ["BoW"],
                     "tokenizers_names": ["Space"]},
                    {"dictionaries_names": ["BiGram"],
                     "feature_calcers": ["BoW"],
                     "tokenizers_names": ["Space"]},
                ]
            },
        },
    )
    model.fit(pool)
    return model


def predict_and_rank(rdf: pd.DataFrame, model: CatBoostRanker, k: int = TOP_K) -> pd.DataFrame:
    """Скоринг + ранжирование, возвращает [user_id, edition_id, rank]."""
    feat_cols = [c for c in NUMERIC_FEATURES if c in rdf.columns]
    text_cols = [c for c in TEXT_FEATURES    if c in rdf.columns]

    pool   = Pool(data=rdf[feat_cols + text_cols], text_features=text_cols,
                  feature_names=feat_cols + text_cols)
    scores = model.predict(pool)

    out = rdf[["user_id", "edition_id"]].copy()
    out["pred"] = scores
    out = (out.sort_values(["user_id", "pred"], ascending=[True, False])
              .groupby("user_id").head(k).copy())
    out["rank"] = (out.groupby("user_id")["pred"]
                      .rank(method="first", ascending=False).astype(int))
    return out[["user_id", "edition_id", "rank"]]


print("CatBoost functions OK")


In [ ]:
# ============================================================
# CELL 9 — Submission Helpers
# ============================================================

def fill_missing(top_k: pd.DataFrame, target_users, edition_feat: pd.DataFrame,
                 k: int = TOP_K) -> pd.DataFrame:
    """Добивает пользователей с < k рекомендациями глобально популярными изданиями."""
    global_top = (edition_feat["pop_log"]
                  .sort_values(ascending=False)
                  .head(k * 3).index.tolist())
    per_user = {u: g for u, g in top_k.groupby("user_id")}
    rows = []

    for uid in target_users:
        grp = per_user.get(uid)
        seen = grp["edition_id"].tolist() if grp is not None else []
        if grp is not None:
            rows.append(grp.head(k))
        need = k - len(seen)
        if need <= 0:
            continue
        extra = [e for e in global_top if e not in set(seen)][:need]
        start = len(seen) + 1
        rows.append(pd.DataFrame({
            "user_id":    uid,
            "edition_id": extra,
            "rank":       range(start, start + len(extra)),
        }))

    return pd.concat(rows, ignore_index=True)


def validate_and_save(sub: pd.DataFrame, output_path: str, k: int = TOP_K) -> pd.DataFrame:
    """Проверяет формат сабмита и сохраняет CSV."""
    errors = []
    tset   = set(targets["user_id"].tolist())

    miss = tset - set(sub["user_id"].unique())
    if miss:
        errors.append(f"{len(miss)} users missing")

    bad_cnt = (sub.groupby("user_id").size() != k).sum()
    if bad_cnt:
        errors.append(f"{bad_cnt} users don't have exactly {k} recs")

    dup_e = (sub.groupby("user_id")["edition_id"].nunique() != k).sum()
    if dup_e:
        errors.append(f"{dup_e} users have duplicate edition_id")

    dup_r = (sub.groupby("user_id")["rank"].nunique() != k).sum()
    if dup_r:
        errors.append(f"{dup_r} users have duplicate rank")

    if errors:
        for e in errors:
            print(f"  ERROR: {e}")
        raise ValueError("Submission invalid")

    sub = sub[["user_id", "edition_id", "rank"]].sort_values(["user_id", "rank"]).reset_index(drop=True)
    sub.to_csv(output_path, index=False)
    print(f"Saved {output_path}  ({len(sub):,} rows)")
    return sub


print("Submission helpers OK")


In [ ]:
# ============================================================
# CELL 9.5 — Сборка данных для Train и Test (MISSING LINK)
# ============================================================
print("--- ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ ---")
# Обучаемся на истории ДО инцидента, предсказываем события ВНУТРИ инцидента
iact_train_hist = interactions[interactions["event_ts"] < T_INCIDENT_START].copy()
iact_train_target = interactions[(interactions["event_ts"] >= T_INCIDENT_START) & 
                                 (interactions["event_type"].isin([1, 2]))].copy()

u_feat_tr  = build_user_features(iact_train_hist, T_TRAIN_SPLIT)
ed_feat_tr = build_edition_features(iact_train_hist, T_TRAIN_SPLIT)
ugp_tr     = build_user_genre_profile(iact_train_hist)
uap_tr     = build_user_author_profile(iact_train_hist)

# Берем для обучения юзеров, у которых есть позитивы в окне инцидента
train_users = iact_train_target['user_id'].unique()

print("Генерация train кандидатов...")
train_cands = generate_candidates(
    train_users, iact_train_hist, ed_feat_tr, ugp_tr, uap_tr, 
    als_model, u2als, e2als, als_edition_ids
)

# Размечаем таргет
target_pairs = set(zip(iact_train_target['user_id'], iact_train_target['edition_id']))
train_cands['target'] = train_cands.apply(
    lambda r: 1 if (r['user_id'], r['edition_id']) in target_pairs else 0, axis=1
)

print("\n--- ПОДГОТОВКА ДАННЫХ ДЛЯ ИНФЕРЕНСА ---")
u_feat_full  = build_user_features(interactions, T_INCIDENT_START)
ed_feat_full = build_edition_features(interactions, T_INCIDENT_START)
ugp_full     = build_user_genre_profile(interactions)
uap_full     = build_user_author_profile(interactions)

test_users = targets['user_id'].unique()

print("Генерация test кандидатов...")
test_cands = generate_candidates(
    test_users, interactions, ed_feat_full, ugp_full, uap_full, 
    als_model, u2als, e2als, als_edition_ids
)

In [ ]:
# ============================================================
# CELL 10 — Формирование обучающей выборки для CatBoost
# ============================================================
import gc

print("Собираем датасет для обучения (Train)...")
train_dataset = build_ranking_dataset(
    cands_df=train_cands, 
    u_feat=u_feat_tr, 
    ed_feat=ed_feat_tr, 
    user_genre_profile=ugp_tr, 
    user_author_profile=uap_tr, 
    editions_info=editions,
    book_genres_df=book_genres # <--- ВАЖНО ДОБАВИТЬ ЭТО
)

# YetiRank требует сортировки по query_id (user_id). Делаем In-Place!
print("Сортировка...")
train_dataset.sort_values('user_id', inplace=True)

# Извлекаем таргет
y_train = train_dataset['target'].values
drop_cols = ['user_id', 'edition_id', 'book_id', 'author_id', 'target']

# Удаляем ненужные колонки In-Place (не создаем новый датафрейм X_train)
print("Очистка признаков...")
train_dataset.drop(columns=[c for c in drop_cols if c in train_dataset.columns], inplace=True)

# Теперь train_dataset — это и есть наш X_train
X_train = train_dataset
features = list(X_train.columns)

print(f"Количество признаков (features): {len(features)}")
print(f"Размер X_train: {X_train.shape}")

# Чистим мусор перед инициализацией Pool-а CatBoost
gc.collect()

In [ ]:
# ============================================================
# CELL 11 — Train CatBoostRanker
# ============================================================
print("=" * 60)
print("STEP 5 | Train CatBoostRanker")
print("=" * 60)

model = train_catboost(X_train, y_train)

imp = pd.Series(
    model.get_feature_importance(type="PredictionValuesChange"),
    index=model.feature_names_,
).sort_values(ascending=False)
print("\nTop-20 feature importances:")
print(imp.head(20).to_string())


In [ ]:
# ============================================================
# CELL 12 — Формирование тестовой выборки и предикт
# ============================================================
print("Собираем датасет для инференса (Test)...")

test_dataset = build_ranking_dataset(
    cands_df=test_cands, 
    u_feat=u_feat_full, 
    ed_feat=ed_feat_full, 
    user_genre_profile=ugp_full, 
    user_author_profile=uap_full, 
    editions_info=editions,
    book_genres_df=book_genres # <--- ВАЖНО ДОБАВИТЬ ЭТО
)

X_test = test_dataset[features]

print("Делаем предсказания CatBoostRanker...")
test_dataset['score'] = cb_model.predict(X_test)

print("Ранжируем кандидатов и выбираем Топ-20...")
test_dataset = test_dataset.sort_values(by=['user_id', 'score'], ascending=[True, False])
test_dataset['rank'] = test_dataset.groupby('user_id').cumcount() + 1

submission = test_dataset[test_dataset['rank'] <= 20][['user_id', 'edition_id', 'rank']]

submission.to_csv('submission.csv', index=False)
print("Готово! Файл submission.csv сохранен и готов к отправке.")


In [ ]:
# ============================================================
# CELL 13 — Fill gaps, validate, save
# ============================================================
print("=" * 60)
print("STEP 10 | Fill + validate + save")
print("=" * 60)

submission = fill_missing(top_k, test_users, ed_feat_full)
submission = validate_and_save(submission, OUTPUT_PATH)

print("\nSample (first 40 rows):")
print(submission.head(40).to_string(index=False))
